# Final Project: Kalshi Market Making in short-dated SPX Events

Chris Mulligan (12502987), George Lord (12243747), Max Zhalilo (12341701)

# Content

0. Abstract
1. Introduction
2. Data
3. Techniques and Theory Details in the Strategy
4. Implementation of our Strategy
5. Result/Analysis

# 0. Abstract

We use statistical modeling & black-scholes to price Kalshi short-dates SPX events. We simulate an automated market making strategy in these events using a simulator that directly interfaces with our trading modules. We delta hedge SPX exposure by trading the SPY ETF. We test the performance of our strategy over 2 months.

# 1. Introduction

In modern financial markets, algorithmic trading and automated market making have become central components of liquidity provision and price discovery. Electronic Market Makers such as Jump & Virtu have begun to expand into prediction markets such as Kalshi & Polymarket to profit from their increasing trading volumes. In this project, we develop and test a fully systematic market-making strategy for short-dated S&P 500 (SPX) event contracts listed on Kalshi. These contracts pay a fixed $1 if a specified event occurs—such as the SPX closing above a certain level-and $0 otherwise. As a result, they can be interpreted as binary options on the terminal value of the SPX index.

The core idea of our strategy is grounded in Black-Scholes derivative pricing theory. A binary call option has a value equal to the discounted risk-neutral probability that the underlying asset finishes above a given strike at expiration. For contracts with 0–1 days to expiration, discounting effects are negligible (??? LETS THINK ABOUT THIS ???), so the contract price closely approximates the market-implied probability of the event occurring. By modeling the terminal distribution of SPX under a log-normal assumption consistent with the Black–Scholes framework, we can estimate these probabilities analytically using closed-form expressions. Specifically, we compute the cumulative distribution function (CDF) implied by current SPX levels, time to expiration, and volatility, with volatility proxied by the VIX index.

Once fair values for binary calls and puts are obtained, we replicate the payoff structures of Kalshi’s SPX “above” and “range” contracts using combinations of these binaries. This replication framework provides a theoretical benchmark price for each listed contract. The strategy then centers its market-making quotes around these fair values and dynamically adjusts bid–ask spreads as a function of market volatility, time to expiration, tick size constraints, exchange fees, and current inventory. In higher-volatility or low-liquidity environments, spreads are widened to compensate for increased uncertainty and adverse selection risk.

Because the fair value of each contract is sensitive to movements in the underlying SPX index, holding inventory in Kalshi contracts introduces directional exposure. To manage this risk, we compute the delta of each replicated binary position with respect to SPX and aggregate the book’s total delta exposure. We then hedge this exposure using the SPY ETF, which closely tracks SPX movements. This delta-hedging process isolates the strategy’s profit and loss primarily to bid–ask spread capture and pricing inefficiencies rather than outright market direction.

We test our strategy in a simulator that mimics Kalshi exchange capability. We've built out our strategy using pricing, market making, hedging, & position management modules, all of which interact with the simulator using our execution engine module. Our objective is to assess whether theoretically grounded probability estimates, combined with disciplined spread management and delta hedging, can generate consistent risk-adjusted returns in short-dated event markets.

# 2. Data

Chris to-do?

# 3. Theory

George to-do?

# 4. Implementation

In our implementation, we created a modular market making bot that currently interfaces with our simulator model. That follows the design below:

![System Architecture](Data/architecture.jpg)

## Data Ingestor

- What it does: Loads and standardizes 1-second Kalshi + market data into structured Polars DataFrames for simulation.
- Takes in: Kalshi, SPX, VIX, SPY parquet files.
- Sends out:
    -  all_df (long-form market + macro data → Simulator)
    - macro_df (SPX/VIX/SPY only → ExecutionEngine)


## Simulator

- What it does: Simulates market activity by iterating through each second and applying last-in-queue fill logic.
- Takes in: all_df (from DataIngestor) + resting quotes & hedge trades (from ExecutionEngine).
- Sends out: FillIntent objects to ExecutionEngine and outputs full per-second log of market behavior + bot behavior.


## Execution Engine

- What it does: Central gateway coordinating pricing, quoting, hedging, delayed execution, and position updates.
- Takes in: SPX/VIX/SPY (from DataIngestor), fill intents (from Simulator), quotes (from MarketMaker), hedge trades (from DeltaHedger).
- Sends out: Fair value requests to Pricer, position updates to PositionManager, quotes to Simulator, and hedge orders to Simulator (applied with 1-second delay).

# 5. Results & Analysis